In [3]:
import time
import types
import numpy as np
import pandas as pd

# Import your Server class (assumes it's defined in servers.py)
import servers

# -----------------------------
# 1) Mock external I/O & model
# -----------------------------
# Mock propagation delays (list per server index)
servers.pickle.load = lambda f: [[0.01, 0.02, 0.03]]  # server 0 has 3 small choices

# Mock reading server_data.csv
servers.pd.read_csv = lambda path: pd.DataFrame({'col': []})

# Mock the computation delay predictor to return deterministic values
# (e.g. 2.5 + (process_id % 3))
servers.computation_delay_regressor.predict_rows = lambda row_num: 2.5 + (int(row_num) % 3)

# -----------------------------
# 2) Small DummyRequest used in demo
# -----------------------------
class DummyRequest:
    def __init__(self, request_id, process_id, message_size=0.0, bandwidth=0.0, load=None):
        self.request_id = int(request_id)
        self.process_id = int(process_id)
        self.message_size = float(message_size)
        self.bandwidth = float(bandwidth)
        if load is None:
            self.load = np.array([1], dtype=int)
        else:
            self.load = np.array(load, dtype=int)

# -----------------------------
# 3) Create server
# -----------------------------
s = servers.Server(server_index=0)

# For readability, make propagation/transmission deterministic (we mocked propagation above,
# but keep transmission simple so total proc_time is driven by predict_rows)
s._get_propogation_delay = lambda: 0.0
s._get_tramission_delay = lambda *args, **kwargs: 0.0

# -----------------------------
# 4) Schedule 4 requests at the same simulated time
# -----------------------------
sim_time = 1000.0
reqs = [DummyRequest(i, process_id=10 + i, message_size=100, bandwidth=1000, load=[1]) for i in range(4)]

print("Scheduling 4 requests at t =", sim_time)
for r in reqs:
    # fetch the original (unscaled) proc time estimate BEFORE scheduling
    orig_proc = s.get_delays(r)
    ok, finish_or_reason, proc = s.schedule_request(r, current_time=sim_time)
    # proc returned by schedule_request should equal orig_proc in this code
    print(f"  req {r.request_id}: accepted={ok}, original_proc_time={orig_proc:.3f}s, "
          f"returned_proc_time={proc:.3f}s, finish_time={finish_or_reason}")

# -----------------------------
# 5) Try to schedule a 5th request (should be rejected)
# -----------------------------
r5 = DummyRequest(99, process_id=99, message_size=100, bandwidth=1000, load=[1])
orig_proc_r5 = s.get_delays(r5)
# schedule_request returns (False, "server full") on rejection (two values)
res = s.schedule_request(r5, current_time=sim_time)
if isinstance(res, tuple) and len(res) == 2:
    ok5, reason5 = res
    print(f"\nAttempt scheduling 5th at same time: accepted={ok5}, reason={reason5}, ")
else:
    # defensive: if spec changed and three values returned
    ok5, finish5, proc5 = res
    print(f"\nAttempt scheduling 5th at same time: accepted={ok5}, finish={finish5}, proc={proc5}")

print("num_requests now:", s.num_requests)

# -----------------------------
# 6) Query time_until_next_free shortly after start
# -----------------------------
ttf = s.time_until_next_free(current_time=sim_time)
print(f"\nTime until next free : {ttf}")

# -----------------------------
# 7) Advance simulated time past finishes and update
# -----------------------------
# advance beyond the largest expected proc (our mocked proc times are small)
advance_to = sim_time + 0.26
print(f"\nAdvancing simulated time to {advance_to} to allow jobs to finish...")
freed = s.update_active_requests(current_time=advance_to)
print("Freed visible requests:", freed)
print("num_requests now: ", s.num_requests)

print("\nActive requests:")

print("Scheduling request 5 again")
r5 = DummyRequest(5, process_id=99, message_size=100, bandwidth=1000, load=[1])
orig_proc_r5 = s.get_delays(r5)
# schedule_request returns (False, "server full") on rejection (two values)
res = s.schedule_request(r5, current_time=sim_time)

if not s.active_requests:
    print("  (none)")
else:
    for a in s.active_requests:
        # Note: in your schedule_request the active entry stores 'proc_time' as the proc used
        req_obj = a["request"]
        start = a["start_time"]
        proc_time = a.get("proc_time", None)
        finish = a["finish_time"]
        print(f"  req {req_obj.request_id}: start={start}, original_proc_time={proc_time:.3f}s, finish_time={finish}")


Scheduling 4 requests at t = 1000.0
  req 0: accepted=True, original_proc_time=3.500s, returned_proc_time=0.350s, finish_time=1000.35
  req 1: accepted=True, original_proc_time=4.500s, returned_proc_time=0.450s, finish_time=1000.45
  req 2: accepted=True, original_proc_time=2.500s, returned_proc_time=0.250s, finish_time=1000.25
  req 3: accepted=True, original_proc_time=3.500s, returned_proc_time=0.350s, finish_time=1000.35

Attempt scheduling 5th at same time: accepted=False, finish=server full, proc=0.25
num_requests now: 4

Time until next free : 0.25

Advancing simulated time to 1000.26 to allow jobs to finish...
Freed visible requests: 1
num_requests now:  3

Active requests:
Scheduling request 5 again
  req 0: start=1000.0, original_proc_time=0.350s, finish_time=1000.35
  req 1: start=1000.0, original_proc_time=0.450s, finish_time=1000.45
  req 3: start=1000.0, original_proc_time=0.350s, finish_time=1000.35
  req 5: start=1000.0, original_proc_time=0.250s, finish_time=1000.25
